In [5]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.size":12,
    "font.family":"DejaVu Sans",
    "axes.linewidth":1.2,
    "grid.alpha":0.4,
    "grid.linestyle":"--",
    "grid.linewidth":0.6,
    "legend.frameon":True,
    "legend.framealpha":0.8,
    "legend.edgecolor":"black",
    "legend.fontsize":10,
    "xtick.direction":"in",
    "ytick.direction":"in",
    "xtick.major.size":5,
    "ytick.major.size":5,
})

STEP=1000
RESULT_DIR=Path("results")
FIG_DIR=Path("fig")
FIG_DIR.mkdir(exist_ok=True)

def load_curve(dataset, model, x_name, y_name, step=STEP):
    """Load x/y curve arrays."""
    x=np.load(RESULT_DIR/f"{dataset}_{model}_{x_name}.npy")[::step]
    y=np.load(RESULT_DIR/f"{dataset}_{model}_{y_name}.npy")[::step]
    return x,y

def save_figure(fig, filename):
    """Save figure."""
    fig.tight_layout()
    fig.savefig(FIG_DIR/filename,dpi=500,bbox_inches="tight")
    plt.close(fig)

def plot_probability_curve(dataset, kind):
    """Plot temporal or spatial probability density."""
    fig,ax=plt.subplots(figsize=(6,4.5))
    if kind=="spatial":
        x_nkf,y_nkf=load_curve(dataset,"neural_neural_neural","dis_xy_train","pdf_spatial_train")
        x_etas,y_etas=load_curve(dataset,"empirical_empirical_empirical","dis_xy_train","pdf_spatial_train")
        mask=x_etas<=x_nkf.max()
        ax.scatter(x_etas[mask],y_etas[mask],s=1,color="#1f77b4",label="ETAS")
        ax.plot(x_nkf,y_nkf,color="#d62728",lw=1.5,label="ST-NKF")
        ax.set_xlabel("Distance (km)")
        out=f"{dataset}-spatial_pattern-loglog.png"
    else:
        x_nkf,y_nkf=load_curve(dataset,"neural_neural_neural","dis_t_train","pdf_temporal_train")
        x_etas,y_etas=load_curve(dataset,"empirical_empirical_empirical","dis_t_train","pdf_temporal_train")
        mask=x_etas<=x_nkf.max()
        ax.plot(x_etas[mask],y_etas[mask],color="#1f77b4",lw=1.5,label="ETAS")
        ax.plot(x_nkf,y_nkf,color="#d62728",lw=1.5,label="ST-NKF")
        ax.set_xlabel("Elapsed Time (day)")
        out=f"{dataset}-temporal_pattern-loglog.png"
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_ylabel("Probability Density")
    ax.grid(True,which="both")
    ax.legend()
    save_figure(fig,out)

def plot_magnitude(dataset):
    """Plot productivity curve."""
    fig,ax=plt.subplots(figsize=(6,4.5))
    x_nkf,y_nkf=load_curve(dataset,"nkf","hist_m_train","kappa_train")
    x_etas,y_etas=load_curve(dataset,"etas","hist_m_train","kappa_train")
    mask=x_etas>=x_nkf.min()
    ax.plot(x_etas[mask],y_etas[mask],color="#1f77b4",lw=1.5,label="ETAS")
    ax.plot(x_nkf,y_nkf,color="#d62728",lw=1.5,label="ST-NKF")
    ax.set_yscale("log")
    ax.set_xlabel("Magnitude")
    ax.set_ylabel("Expected Number of Offspring")
    ax.grid(True)
    ax.legend(loc="upper left")
    save_figure(fig,f"{dataset}-kappa_pattern.png")



In [ ]:
datasets=["SCEDC_20","SCEDC_25","SCEDC_30","ComCat","SaltonSea","SanJac","WHITE"]
for ds in datasets:
    plot_probability_curve(ds,"spatial")
    plot_probability_curve(ds,"temporal")
    plot_magnitude(ds)
